<h2>What UCS is</h2>
<p>
UCS (Uniform Cost Search) is a search method that finds the cheapest path from a start location to a goal location. It considers the cost (weight) of each path and always expands the path with the lowest total cost first.
</p>

<h2>UCS Steps (Simple)</h2>
<p>
I. Start from the initial location (start node).<br>
II. Put the start node into a priority queue with cost 0.<br>
III. Remove the node with the lowest cost from the queue.<br>
IV. Check if it is the goal. If yes, stop.<br>
V. If it is not the goal, explore all its neighbors.<br>
VI. Add each neighbor to the queue with updated total cost.<br>
VII. Repeat steps III to VI until the goal is found.
</p>



In [3]:
import heapq
from graph import graph

# -----------------------------
# UCS FUNCTION
# -----------------------------
def ucs(start, goal):
    # priority queue: (cost, path)
    queue = [(0, [start])]

    visited = {}

    while queue:
        cost, path = heapq.heappop(queue)
        node = path[-1]

        # GOAL CHECK
        if node == goal:
            return path, cost

        # EXPAND ONLY IF BETTER COST
        if node not in visited or cost < visited[node]:
            visited[node] = cost

            for neighbor, weight in graph[node]:
                new_cost = cost + weight
                new_path = path + [neighbor]
                heapq.heappush(queue, (new_cost, new_path))

    return None, None


# -----------------------------
# RUN UCS
# -----------------------------
start = "Campus Center"
goal = "Physics Lab"

path, cost = ucs(start, goal)

# -----------------------------
# OUTPUT
# -----------------------------
print("START:", start)
print("GOAL :", goal)
print("----------------------")

if path:
    print("PATH FOUND:")
    print(" -> ".join(path))
    print("TOTAL COST:", cost)
else:
    print("No path found")

START: Campus Center
GOAL : Physics Lab
----------------------
PATH FOUND:
Campus Center -> Learning Area -> LH01 -> LH02 -> LH03 -> LH04 -> Physics Lab
TOTAL COST: 8


In [2]:
import tkinter as tk
from tkinter import ttk
import heapq
import time

from graph import graph

# Node positions for visualization
pos = {
    "Campus Center": (400, 30),
    "Main Library": (180, 120),
    "Admin Office": (350, 120),
    "Learning Area": (550, 120),
    "Café": (750, 120),
    "Female Library": (130, 220),
    "Digital Library": (260, 220),
    "HR Office": (370, 220),
    "Freshman Office": (470, 220),
    "Staff Lounge": (560, 220),
    "LH01": (640, 220),
    "CL01": (720, 220),
    "Male Lounge": (810, 220),
    "LH02": (640, 320),
    "LH03": (640, 400),
    "LH04": (640, 480),
    "Physics Lab": (640, 560),
    "Chemistry Lab": (640, 620),
    "Biology Lab": (640, 680),
    "Sports Center": (640, 740),
    "CL02": (720, 320),
    "CL03": (720, 400),
    "CL04": (720, 480),
    "Female Lounge": (900, 220)
}

def ucs(start, goal, callback=None):
    queue = [(0, [start])]
    visited = {}
    expanded_nodes = []
    
    while queue:
        queue.sort(key=lambda x: x[0])
        cost, path = queue.pop(0)
        node = path[-1]
        
        expanded_nodes.append(node)
        if callback:
            callback(node, cost, path, "exploring")
            time.sleep(0.5)
        
        if node == goal:
            if callback:
                callback(node, cost, path, "goal")
            return path, cost, expanded_nodes
        
        if node not in visited or cost < visited[node]:
            visited[node] = cost
            
            for neighbor, weight in graph[node]:
                new_cost = cost + weight
                new_path = path + [neighbor]
                queue.append((new_cost, new_path))
                if callback:
                    callback(neighbor, new_cost, new_path, "queued")
                    time.sleep(0.3)
    
    return None, None, expanded_nodes

class UCSVisualization:
    def __init__(self, root):
        self.root = root
        self.root.title("UCS - Uniform Cost Search Campus Navigation")
        self.root.geometry("1200x850")
        self.root.configure(bg="#1a1a2e")
        
        main_frame = tk.Frame(root, bg="#1a1a2e")
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(main_frame, width=900, height=780, bg="#0f0f1a", highlightthickness=0)
        self.canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        control_frame = tk.Frame(main_frame, bg="#16213e", width=300)
        control_frame.pack(side=tk.RIGHT, fill=tk.Y, padx=5, pady=10)
        control_frame.pack_propagate(False)
        
        title = tk.Label(control_frame, text="UNIFORM COST SEARCH", font=("Arial", 16, "bold"), bg="#16213e", fg="#e94560")
        title.pack(pady=15)
        
        desc = tk.Label(control_frame, text="Finds the path with minimum total cost", font=("Arial", 9), bg="#16213e", fg="#aaaaaa")
        desc.pack(pady=5)
        
        start_frame = tk.Frame(control_frame, bg="#16213e")
        start_frame.pack(pady=10, fill=tk.X, padx=20)
        
        tk.Label(start_frame, text="START:", font=("Arial", 11, "bold"), bg="#16213e", fg="#00ff88").pack(anchor=tk.W)
        self.start_var = tk.StringVar(value="Campus Center")
        self.start_combo = ttk.Combobox(start_frame, textvariable=self.start_var, values=list(pos.keys()), font=("Arial", 10))
        self.start_combo.pack(fill=tk.X, pady=5)
        
        goal_frame = tk.Frame(control_frame, bg="#16213e")
        goal_frame.pack(pady=10, fill=tk.X, padx=20)
        
        tk.Label(goal_frame, text="GOAL:", font=("Arial", 11, "bold"), bg="#16213e", fg="#e94560").pack(anchor=tk.W)
        self.goal_var = tk.StringVar(value="Physics Lab")
        self.goal_combo = ttk.Combobox(goal_frame, textvariable=self.goal_var, values=list(pos.keys()), font=("Arial", 10))
        self.goal_combo.pack(fill=tk.X, pady=5)
        
        self.run_button = tk.Button(control_frame, text="▶ RUN UCS", font=("Arial", 12, "bold"), bg="#e94560", fg="white", command=self.run_search)
        self.run_button.pack(pady=15, padx=20, fill=tk.X)
        
        self.reset_button = tk.Button(control_frame, text="⟳ RESET", font=("Arial", 11), bg="#5c5c8a", fg="white", command=self.reset_canvas)
        self.reset_button.pack(pady=5, padx=20, fill=tk.X)
        
        info_frame = tk.Frame(control_frame, bg="#1a1a2e", relief=tk.RAISED, bd=1)
        info_frame.pack(pady=15, padx=20, fill=tk.X)
        
        self.cost_label = tk.Label(info_frame, text="TOTAL COST: --", font=("Arial", 11, "bold"), bg="#1a1a2e", fg="#ffd700")
        self.cost_label.pack(pady=8)
        
        self.steps_label = tk.Label(info_frame, text="STEPS: --", font=("Arial", 11), bg="#1a1a2e", fg="#ffffff")
        self.steps_label.pack(pady=5)
        
        self.expanded_label = tk.Label(info_frame, text="NODES EXPANDED: --", font=("Arial", 10), bg="#1a1a2e", fg="#aaaaaa")
        self.expanded_label.pack(pady=5)
        
        path_frame = tk.Frame(control_frame, bg="#16213e")
        path_frame.pack(pady=10, padx=20, fill=tk.BOTH, expand=True)
        
        tk.Label(path_frame, text="FOUND PATH:", font=("Arial", 11, "bold"), bg="#16213e", fg="#00ff88").pack(anchor=tk.W)
        
        self.path_text = tk.Text(path_frame, height=15, width=32, bg="#0f0f1a", fg="#e0e0ff", font=("Courier", 9), wrap=tk.WORD)
        self.path_text.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.draw_graph()
    
    def draw_graph(self):
        self.canvas.delete("all")
        for node in graph:
            if node not in pos:
                continue
            x, y = pos[node]
            text_width = len(node) * 7
            rect_x1 = x - text_width//2 - 10
            rect_x2 = x + text_width//2 + 10
            rect_y1 = y - 12
            rect_y2 = y + 12
            
            self.canvas.create_rectangle(rect_x1, rect_y1, rect_x2, rect_y2, fill="#2d2d44", outline="#5c5c8a", width=2, tags=f"node_{node}")
            self.canvas.create_text(x, y, text=node, font=("Arial", 8, "bold"), fill="#e0e0ff", tags=f"text_{node}")
            
            for neighbor, weight in graph[node]:
                if neighbor in pos:
                    nx, ny = pos[neighbor]
                    self.canvas.create_line(x, y, nx, ny, fill="#3d3d5c", width=2, tags=f"edge_{node}_{neighbor}")
                    mid_x, mid_y = (x + nx)//2, (y + ny)//2
                    self.canvas.create_text(mid_x, mid_y - 5, text=str(weight), font=("Arial", 8), fill="#ffd700", tags=f"weight_{node}_{neighbor}")
    
    def reset_canvas(self):
        self.draw_graph()
        self.cost_label.config(text="TOTAL COST: --")
        self.steps_label.config(text="STEPS: --")
        self.expanded_label.config(text="NODES EXPANDED: --")
        self.path_text.delete(1.0, tk.END)
        self.run_button.config(state=tk.NORMAL, text="▶ RUN UCS")
    
    def update_node_color(self, node, color, delay=0.2):
        if node not in pos:
            return
        x, y = pos[node]
        text_width = len(node) * 7
        rect_x1 = x - text_width//2 - 10
        rect_x2 = x + text_width//2 + 10
        rect_y1 = y - 12
        rect_y2 = y + 12
        
        self.canvas.delete(f"node_{node}")
        self.canvas.create_rectangle(rect_x1, rect_y1, rect_x2, rect_y2, fill=color, outline="#7a7aaa", width=2, tags=f"node_{node}")
        self.canvas.create_text(x, y, text=node, font=("Arial", 8, "bold"), fill="#ffffff", tags=f"text_{node}")
        self.root.update()
        time.sleep(delay)
    
    def update_path_display(self, path, total_cost):
        self.path_text.delete(1.0, tk.END)
        for i, node in enumerate(path):
            if i == 0:
                self.path_text.insert(tk.END, f"🏁 START: {node}\n")
            elif i == len(path) - 1:
                self.path_text.insert(tk.END, f"🎯 GOAL: {node}\n")
            else:
                self.path_text.insert(tk.END, f"  → {node}\n")
        
        self.cost_label.config(text=f"TOTAL COST: {total_cost}")
        self.steps_label.config(text=f"STEPS: {len(path) - 1}")
    
    def search_callback(self, node, cost, path, status):
        if status == "exploring":
            self.update_node_color(node, "#e94560", 0.15)
        elif status == "queued":
            self.update_node_color(node, "#ffaa44", 0.1)
        elif status == "goal":
            total_cost = cost
            for i, n in enumerate(path):
                self.update_node_color(n, "#00ff88", 0.1)
            self.update_path_display(path, total_cost)
            self.expanded_label.config(text=f"NODES EXPANDED: {len(set(path))}")
            self.run_button.config(state=tk.NORMAL, text="✓ DONE")
    
    def run_search(self):
        self.reset_canvas()
        self.run_button.config(state=tk.DISABLED, text="⟳ SEARCHING...")
        
        start = self.start_var.get()
        goal = self.goal_var.get()
        
        if start not in graph or goal not in graph:
            messagebox.showerror("Error", "Invalid start or goal location!")
            self.run_button.config(state=tk.NORMAL, text="▶ RUN UCS")
            return
        
        self.root.update()
        
        path, cost, expanded = ucs(start, goal, self.search_callback)
        
        if not path:
            self.path_text.delete(1.0, tk.END)
            self.path_text.insert(tk.END, "❌ NO PATH FOUND!\n")
            self.path_text.insert(tk.END, f"From '{start}' to '{goal}'")
            self.cost_label.config(text="TOTAL COST: ∞")
            self.run_button.config(state=tk.NORMAL, text="▶ RUN UCS")

if __name__ == "__main__":
    root = tk.Tk()
    app = UCSVisualization(root)
    root.mainloop()